# Baga AI — модель цены аренды

Предсказываем справедливую месячную аренду по характеристикам квартиры и отвечаем на
вопрос «не выше ли цена, чем у похожих».

Порядок разделов — это и есть порядок работы:

```
1. загрузка      krisha.db / listings_raw.parquet
2. чистка        единая схема, отсев мусора, районы из адреса
3. фильтрация    ВЫРЕЗАЕМ ЦЕНУ ИЗ ОПИСАНИЯ  ← без этого всё остальное бессмысленно
4. признаки      адрес -> район/мкр/улица, флаги из текста, числа
5. разбиение     фолды: страты по районам, группы по дублям
6. бейзлайн      медиана ₸/м² по району × комнатам
7. CatBoost      квантили 10/50/90 на log(price), лестница наборов признаков
8. калибровка    конформная поправка интервала (CQR)
9. итоги         таблица, сегменты, важность признаков
10. применение   вердикт по объявлению + сохранение модели
```

Главная ловушка этих данных: **у 60% объявлений цена написана в тексте описания**,
у 57.6% — ровно та, которую мы предсказываем. Модель, обученная на сыром тексте,
покажет великолепную метрику и развалится на новых данных. Раздел 3 именно про это.

## 1. Загрузка

In [ ]:
import json, os, re, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

LISTING_COLS = ["id", "url", "deal_type", "rent_period", "price_kzt", "rooms", "area_total",
                "area_living", "area_kitchen", "floor", "floors_total", "year_built", "building_type",
                "complex_name", "city", "district", "address", "lat", "lon", "description",
                "furniture", "bathroom", "balcony", "renovation_text", "published_at",
                "photos_count", "duplicate_group_id"]


def load_raw(roots=("data", "../data", "/kaggle/input", "..", ".")) -> pd.DataFrame:
    """Берём listings_raw.parquet, если он есть, иначе читаем krisha.db напрямую."""
    for root in roots:
        p = Path(root)
        if not p.exists():
            continue
        for f in p.rglob("listings_raw.parquet"):
            print("parquet:", f)
            return pd.read_parquet(f)
        for f in p.rglob("krisha.db"):
            print("sqlite:", f)
            con = sqlite3.connect(f"file:{f}?mode=ro&immutable=1", uri=True)
            return pd.read_sql(f"SELECT {', '.join(LISTING_COLS)} FROM listings", con)
    raise FileNotFoundError("не найдены ни listings_raw.parquet, ни krisha.db")


raw = load_raw()
print(raw.shape)
raw.head(3)[["id", "price_kzt", "rooms", "area_total", "city", "district", "address"]]

## 2. Чистка: единая схема и отсев мусора

Три вещи. Переименование колонок парсера в наши. Приведение к числам («45 000 ₸» → 45000).
Отсев нереального: аренда за пределами 1 000–30 000 ₸/м² в месяц — это опечатки
(«35 000 ₸ за 80 м²»), а не рынок.

Плюс две вещи, без которых потом не работают ни фильтры, ни модель:
названия городов и районов переводим на русский (в базе они латиницей),
а район добираем из адреса — в колонке он есть у 39%, в адресе у 55%, вместе 94%.

In [ ]:
RENAME = {"id": "listing_id", "price_kzt": "price", "area_total": "area",
          "area_kitchen": "kitchen_area", "renovation_text": "condition"}
PPM_MIN, PPM_MAX = 1_000, 30_000          # ₸ за м² в месяц
AREA_MIN, AREA_MAX = 8, 1000
CITY_NAMES = {"almaty": "алматы", "kaskelen": "каскелен", "astana": "астана"}
DISTRICT_NAMES = {"bostandyk": "бостандыкский", "alatau": "алатауский", "almalin": "алмалинский",
                  "medeu": "медеуский", "auezov": "ауэзовский", "nauryzbay": "наурызбайский",
                  "turksib": "турксибский", "zhetysu": "жетысуский"}
DISTRICT_RE = re.compile(r"([А-Яа-яЁё\-]+)\s+р-н")


def to_number(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    return pd.to_numeric(s.astype("string").str.replace(",", ".", regex=False)
                         .str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")


def district_ru(v):
    if not isinstance(v, str):
        return pd.NA
    for key, ru in DISTRICT_NAMES.items():
        if key in v:
            return f"{ru} р-н"
    return v


def district_from_address(a):
    m = DISTRICT_RE.search(a) if isinstance(a, str) else None
    return f"{m.group(1).lower()} р-н" if m else pd.NA


def normalize(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.rename(columns=RENAME).copy()
    df["listing_id"] = df.listing_id.astype(str)
    for c in ["price", "area", "rooms", "floor", "floors_total", "kitchen_area", "lat", "lon"]:
        if c in df:
            df[c] = to_number(df[c])
    for c in ["city", "district", "condition", "building_type"]:
        if c in df:
            df[c] = df[c].astype("string").str.strip().str.lower().replace("", pd.NA)
    df["city"] = df.city.replace(CITY_NAMES)
    df["district"] = (df.district.map(district_ru).astype("string")
                      .fillna(df.address.map(district_from_address).astype("string"))
                      .fillna("не указан"))

    n0 = len(df)
    if "rent_period" in df:
        df = df[df.rent_period.fillna("month") == "month"]      # посуточная — другой рынок
    df = df[~df.duplicated("listing_id")]
    df = df[df.price.notna() & (df.price > 0)]
    df = df[df.area.between(AREA_MIN, AREA_MAX)]
    df = df[df.rooms.between(0, 20)]
    ppm = df.price / df.area
    df = df[ppm.between(PPM_MIN, PPM_MAX)]
    print(f"строк: {n0} -> {len(df)} (отсеяно {n0 - len(df)})")

    df = df.reset_index(drop=True)
    df["rooms"] = df.rooms.astype(int)
    df["price_per_m2"] = df.price / df.area
    df["floor_rel"] = df.floor / df.floors_total
    df["is_first_floor"] = (df.floor == 1).astype(float).where(df.floor.notna())
    df["is_last_floor"] = (df.floor == df.floors_total).astype(float).where(df.floor.notna())
    return df


df = normalize(raw)
print("\nрайон известен у", f"{(df.district != 'не указан').mean():.0%}")
print(df.district.value_counts().to_string())

## 3. Фильтрация описаний: вырезаем цену

Цену пишут как угодно: «750 000 ₸ за месяц», «Аренда 200000 + ком.услуги»,
«190000 тысяч в месяц», «350000 тнг». Перебирать варианты бесполезно, поэтому правило
простое: **любое число от 10 000 — это деньги**. Все остальные числа в объявлении мельче:
площадь до 980, этаж до 20, год до 2026, потолки 2.5–3.5.

Ещё отрезаем интерфейсный хвост сайта («В Избранное · Сохранить в подборку»).

`assert_no_price_leak` берёт настоящую цену и ищет её цифры в очищенном тексте. Если
найдёт больше чем у 1% — обучение падает. Эта проверка уже один раз поймала утечку.

In [ ]:
MONEY_MIN = 10_000
NUM_SEQ_RE = re.compile(r"\d[\d\s .,]*\d|\d")
TAIL_RE = re.compile(r"(Хозяин недвижимости|В Избранное|Сохранить в подборку).*$", re.S)


def mask_money(m):
    digits = re.sub(r"\D", "", m.group(0))
    return " цена " if len(digits) >= 5 and int(digits) >= MONEY_MIN else m.group(0)


def clean_description(text) -> str:
    if not isinstance(text, str):
        return ""
    text = TAIL_RE.sub(" ", text)
    text = NUM_SEQ_RE.sub(mask_money, text)
    return re.sub(r"\s+", " ", text).strip()


def assert_no_price_leak(texts, prices):
    share = float(np.mean([str(int(p)) in t.replace(" ", "")
                           for t, p in zip(texts.fillna(""), prices.fillna(0))]))
    if share > 0.01:
        raise AssertionError(f"цена осталась в тексте у {share:.1%} объявлений")
    print(f"утечка цены в тексте: {share:.2%} — чисто")


desc = df.description.map(clean_description)
r = df.iloc[0]
print("БЫЛО: ", r.description[:220], "\n")
print("СТАЛО:", clean_description(r.description)[:220], "\n")
assert_no_price_leak(desc, df.price)

## 4. Признаки

Три источника:

- **числа из таблицы** — площадь, комнаты, этаж, относительный этаж, площадь кухни;
- **адрес** (заполнен у 100%) — район, микрорайон, улица, пересечение улиц. Для аренды
  локация важнее всего остального;
- **описание** — флаги: мебель, техника, интернет, депозит, охрана, парковка, тип санузла.

Год постройки и тип дома парсер почти не достаёт (1–3%), поэтому их в признаках
фактически нет. Это известная дыра, закрывается только повторным сбором детальных страниц.

In [ ]:
FLAGS = {
    "furnished": r"меблирован", "no_furniture": r"без мебели",
    "appliances": r"(?:стиральн|холодильник|посудомо)", "internet": r"интернет",
    "deposit": r"(?:депозит|предоплат)", "kids_pets_ok": r"(?:можно с детьми|с животными)",
    "new_building": r"(?:новостройк|жк\s)", "renovation": r"ремонт",
    "parking": r"(?:парковк|паркинг)", "security": r"(?:охран|консьерж|видеонаблюден)",
    "separate_bath": r"санузел\s*раздельн", "combined_bath": r"санузел\s*совмещ",
}
NUMS = {"kitchen_area_txt": r"кухня\s*(\d+[.,]?\d*)\s*м",
        "living_area_txt": r"жил\.?\s*площадь\s*(\d+[.,]?\d*)",
        "ceiling": r"потолки\s*(\d[.,]\d)", "year_txt": r"(\d{4})\s*г\.?\s*п"}
HOUSE_RE = re.compile(r"\s+\d+[а-я]?(/\d+)?$", re.I)


def parse_address(addr) -> dict:
    """«Медеуский р-н, мкр Самал-2, Аль-Фараби 17 — Достык»
    -> мкр самал-2 / аль-фараби / достык (часть после тире — пересечение улиц)."""
    out = {"mkr": pd.NA, "street": pd.NA, "cross_street": pd.NA}
    if not isinstance(addr, str):
        return out
    head, _, tail = addr.partition("—")
    if tail.strip():
        out["cross_street"] = tail.strip().lower()
    parts = [p.strip().lower() for p in head.split(",") if p.strip()]
    mkr = [p for p in parts if p.startswith(("мкр", "мк-р", "микрорайон"))]
    street = [p for p in parts if p not in mkr and "р-н" not in p]
    if mkr:
        out["mkr"] = HOUSE_RE.sub("", mkr[0]).strip()
    if street:
        out["street"] = HOUSE_RE.sub("", street[-1]).strip()
    return out


NUM_COLS = ["area", "rooms", "floor", "floors_total", "floor_rel", "is_first_floor", "is_last_floor",
            "kitchen_area", "area_per_room", "photos_count", "lat", "lon"]
DESC_NUM = list(NUMS) + ["desc_len"]
CAT_COLS = ["city", "district", "mkr", "street", "cross_street", "condition"]
FLAG_COLS = list(FLAGS)


def build_features(df: pd.DataFrame, desc: pd.Series) -> pd.DataFrame:
    f = pd.DataFrame(index=df.index)
    for c in NUM_COLS:
        f[c] = pd.to_numeric(df[c], errors="coerce") if c in df else np.nan
    f["area_per_room"] = df.area / df.rooms.replace(0, np.nan)
    f["desc_len"] = desc.str.len()
    for name, pat in NUMS.items():
        f[name] = pd.to_numeric(desc.str.extract(pat, flags=re.I)[0].str.replace(",", ".", regex=False),
                                errors="coerce")
    for name, pat in FLAGS.items():
        f[name] = desc.str.contains(pat, case=False, regex=True).astype(int)
    addr = pd.DataFrame([parse_address(a) for a in df.address], index=df.index)
    for c in CAT_COLS:
        src = addr[c] if c in addr else df.get(c)
        f[c] = pd.Series(src, index=df.index).astype("string").fillna("не указан")
    return f


X = build_features(df, desc)
y = df.price.to_numpy()
print("признаков:", X.shape[1])
print("\nзаполненность числовых:")
print(X[NUM_COLS + DESC_NUM].notna().mean().round(2).sort_values().to_string())
print("\nфлаги (доля объявлений):")
print(X[FLAG_COLS].mean().round(2).to_string())

## 5. Разбиение

Два требования одновременно:

- **страты по районам** — район главный фактор цены, он должен быть представлен везде
  в одинаковой пропорции;
- **группы по дублям** — одна квартира от трёх риелторов не должна попасть и в обучение,
  и в проверку, иначе модель просто узнает знакомую квартиру.

Метрику считаем только на невиданных данных. Чтобы при этом получить честное
предсказание для **всех** объявлений, используем out-of-fold: пять моделей, каждая
предсказывает свой отложенный кусок.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5


def add_folds(df, n_splits=N_SPLITS, seed=0):
    strata = (df.city.astype(str) + " / " + df.district.astype(str)).to_numpy()
    counts = pd.Series(strata).value_counts()
    rare = set(counts[counts < n_splits].index)
    strata = np.array(["прочее" if s in rare else s for s in strata])
    groups = (df.duplicate_group_id.astype("string").fillna(pd.Series(df.index.astype(str)))
              if "duplicate_group_id" in df else pd.Series(df.index.astype(str))).to_numpy()

    folds = np.full(len(df), -1)
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (_, test_idx) in enumerate(cv.split(df, strata, groups)):
        folds[test_idx] = k
    return folds, groups


df["fold"], groups = add_folds(df)
leaked = pd.Series(groups).groupby(groups).apply(lambda g: df.loc[g.index, "fold"].nunique() > 1).sum()
print(f"групп, размазанных по фолдам: {leaked} (должно быть 0)")
print("размеры фолдов:", df.fold.value_counts().sort_index().to_dict())
print("\nдоля района внутри фолда:")
print(pd.crosstab(df.district, df.fold, normalize="columns").round(3).to_string())

## 6. Бейзлайн: медиана ₸/м² по району и комнатам

То, что сделал бы человек без машинного обучения: посмотреть, почём сдают похожие,
и умножить на площадь. Перцентили 10/50/90 берём по той же группе — получается и оценка,
и интервал.

Если CatBoost это не обгонит, значит проблема в признаках, а не в модели.

In [ ]:
QUANTILES = (0.1, 0.5, 0.9)


def baseline_quantiles(train, test):
    t = train.assign(ppm=train.price / train.area)
    keys = [["city", "district", "rooms"], ["city", "district"], ["city"]]
    out = np.full((len(test), 3), np.nan)
    for k in keys:                                    # мелкая группа -> откат на район, потом город
        tab = t.groupby(k).ppm.quantile(list(QUANTILES)).unstack()
        idx = pd.MultiIndex.from_frame(test[k]) if len(k) > 1 else pd.Index(test[k[0]])
        out = np.where(np.isnan(out), tab.reindex(idx).to_numpy(), out)
    out = np.where(np.isnan(out), np.nanmedian(t.ppm), out)
    return out * test.area.to_numpy()[:, None]


def metrics(y, pred):
    p10, p50, p90 = np.sort(pred, axis=1).T        # три квантиля иногда пересекаются — сортируем
    return {"MAE, ₸": int(np.mean(np.abs(y - p50))),
            "MAPE, %": round(float(np.mean(np.abs(y - p50) / y) * 100), 1),
            "покрытие 10-90, %": round(float(np.mean((y >= p10) & (y <= p90)) * 100), 1),
            "ширина, % от цены": round(float(np.median((p90 - p10) / p50) * 100))}


pred_base = np.full((len(df), 3), np.nan)
for k in range(N_SPLITS):
    tr, te = df[df.fold != k], df[df.fold == k]
    pred_base[te.index] = baseline_quantiles(tr, te)
print(metrics(y, pred_base))

## 7. CatBoost: квантильная регрессия

Обычная регрессия предсказывает среднее. Нам нужен интервал, поэтому функция потерь —
pinball: при α = 0.1 недооценка штрафуется в 9 раз слабее переоценки, и модели выгодно
держаться низко, то есть выучить нижнюю границу.

Учим на `log(price)`: у цен длинный правый хвост, а ошибка становится относительной.

Лестница наборов признаков нужна, чтобы на защите сказать не «мы взяли CatBoost», а
«вот сколько дал каждый блок признаков».

In [ ]:
from catboost import CatBoostRegressor, Pool

FEATURE_SETS = {
    "catboost база":     NUM_COLS + [c for c in CAT_COLS if c != "condition"],
    "+ ремонт":          NUM_COLS + CAT_COLS,
    "+ признаки текста": NUM_COLS + DESC_NUM + FLAG_COLS + CAT_COLS,
}


def fit_catboost(train_idx, cols, iterations=1200):
    cats = [c for c in cols if c in CAT_COLS]
    xtr = X.loc[train_idx, cols].copy()
    for c in cats:
        xtr[c] = xtr[c].astype(str)
    model = CatBoostRegressor(
        loss_function="MultiQuantile:alpha=" + ",".join(str(q) for q in QUANTILES),
        iterations=iterations, depth=6, learning_rate=0.05, verbose=0, random_seed=0)
    model.fit(Pool(xtr, np.log(y[train_idx]), cat_features=cats))
    return model, cats


def predict(model, cats, idx, cols):
    xte = X.loc[idx, cols].copy()
    for c in cats:
        xte[c] = xte[c].astype(str)
    return np.exp(model.predict(Pool(xte, cat_features=cats)))


preds = {"бейзлайн: район × м²": pred_base}
for name in FEATURE_SETS:
    preds[name] = np.full((len(df), 3), np.nan)

for k in range(N_SPLITS):
    tr_idx = df.index[df.fold != k]
    te_idx = df.index[df.fold == k]
    for name, cols in FEATURE_SETS.items():
        model, cats = fit_catboost(tr_idx, cols)
        preds[name][te_idx] = predict(model, cats, te_idx, cols)
    print(f"фолд {k + 1}/{N_SPLITS}")

pd.DataFrame({n: metrics(y, p) for n, p in preds.items()}).T

## 8. Конформная калибровка интервала

У квантильной регрессии покрытие вышло 66% вместо обещанных 80%: коридор слишком узкий,
и каждая третья нормальная квартира получила бы вердикт «дороже похожих».

CQR это чинит без переобучения модели: часть обучающих данных откладываем, смотрим,
насколько сильно правда вылезает за границы, и расширяем коридор ровно на столько.
В логарифме сдвиг границы — это умножение в тенге, поэтому дорогие квартиры получают
пропорционально более широкий интервал.

In [ ]:
BEST_SET = "+ признаки текста"
CALIB_FRAC = 0.25


def conformal_q(train_idx, cols, seed=0):
    """Насколько надо расширить коридор, чтобы покрытие стало 80%."""
    calib = pd.Index(pd.Series(train_idx).sample(frac=CALIB_FRAC, random_state=seed).to_numpy())
    fit = train_idx.difference(calib)
    model, cats = fit_catboost(fit, cols)
    p = np.sort(predict(model, cats, calib, cols), axis=1)
    yy = np.log(y[calib])
    scores = np.maximum(np.log(p[:, 0]) - yy, yy - np.log(p[:, 2]))
    return float(np.quantile(scores, 0.8))


preds["+ конформная калибровка"] = np.full((len(df), 3), np.nan)
q_by_fold = []
for k in range(N_SPLITS):
    tr_idx, te_idx = df.index[df.fold != k], df.index[df.fold == k]
    q = conformal_q(tr_idx, FEATURE_SETS[BEST_SET])
    q_by_fold.append(q)
    model, cats = fit_catboost(tr_idx, FEATURE_SETS[BEST_SET])
    p = np.sort(predict(model, cats, te_idx, FEATURE_SETS[BEST_SET]), axis=1)
    preds["+ конформная калибровка"][te_idx] = np.stack(
        [p[:, 0] * np.exp(-q), p[:, 1], p[:, 2] * np.exp(q)], axis=1)
    print(f"фолд {k + 1}: расширение ×{np.exp(q):.2f}")

table = pd.DataFrame({n: metrics(y, p) for n, p in preds.items()}).T
table

## 9. Итоги: сегменты и важность признаков

In [ ]:
final = np.sort(preds["+ конформная калибровка"], axis=1)
seg = df.assign(p50=final[:, 1], hit=(y >= final[:, 0]) & (y <= final[:, 2]),
                ape=np.abs(y - final[:, 1]) / y * 100)

print("по городам:")
print(seg.groupby("city").agg(объявлений=("price", "size"), MAPE=("ape", "mean"),
                              покрытие=("hit", "mean")).round(2).to_string())
seg["сегмент"] = pd.qcut(seg.price, 4, labels=["дешёвые", "ниже медианы", "выше медианы", "дорогие"])
print("\nпо цене:")
print(seg.groupby("сегмент", observed=True).agg(объявлений=("price", "size"), MAPE=("ape", "mean"),
                                                покрытие=("hit", "mean")).round(2).to_string())

model_full, cats_full = fit_catboost(df.index, FEATURE_SETS[BEST_SET])
imp = pd.Series(model_full.get_feature_importance(), index=FEATURE_SETS[BEST_SET]).sort_values(ascending=False)
print("\nважность признаков (топ-15):")
print(imp.head(15).round(1).to_string())

## 10. Применение: вердикт и сохранение модели

Вердикт формулируем от интервала, а не от точечной оценки, и в терминах объявлений,
а не сделок: «цена выше диапазона похожих объявлений». Рядом всегда показываем аналоги —
пользователь должен видеть, на чём основано решение.

In [ ]:
def verdict(price, p10, p50, p90, n_similar):
    if n_similar < 10:
        note = " (похожих объявлений мало, оценка ненадёжна)"
    else:
        note = ""
    if price < p10:
        return "ниже диапазона похожих объявлений" + note
    if price > p90:
        return "выше диапазона похожих объявлений" + note
    return "в диапазоне похожих объявлений" + note


def comparables(row, k=5):
    """Похожие объявления со смягчением условий.

    Для редких квартир (12 комнат, 980 м²) точных аналогов не существует, и жёсткий
    фильтр возвращает пустоту — вердикт без единого обоснования. Поэтому условия
    ослабляются по шагам, а на выходе видно, на каком шаге нашлось и сколько их было:
    мало аналогов = предупреждение пользователю, а не молчаливая уверенность.
    """
    same = df.listing_id != row.listing_id
    steps = [
        ("район + комнаты + площадь ±20%",
         (df.district == row.district) & (df.rooms == row.rooms) & df.area.between(row.area * .8, row.area * 1.2)),
        ("район + комнаты",
         (df.district == row.district) & (df.rooms == row.rooms)),
        ("город + комнаты + площадь ±30%",
         (df.city == row.city) & (df.rooms == row.rooms) & df.area.between(row.area * .7, row.area * 1.3)),
        ("город + площадь ±30%",
         (df.city == row.city) & df.area.between(row.area * .7, row.area * 1.3)),
    ]
    for label, m in steps:
        m = m & same
        if m.sum() >= 3:
            out = df[m].assign(d=(df[m].area - row.area).abs()).nsmallest(k, "d")
            return out[["listing_id", "price", "area", "floor", "district", "condition"]], label, int(m.sum())
    # последний шаг: показать хоть что-то, но честно пометив, что это не аналоги
    pool = df[same & (df.city == row.city)]
    out = pool.assign(d=(pool.area - row.area).abs()).nsmallest(k, "d")
    return out[["listing_id", "price", "area", "floor", "district", "condition"]], "похожих нет, ближайшие по площади", 0


def show_verdict(i):
    row = df.loc[i]
    comps, how, n = comparables(row)
    print(f"объявление {row.listing_id}: {row.rooms}-комн, {row.area} м², {row.district}, "
          f"{row.floor}/{row.floors_total} этаж")
    print(f"просят:   {int(row.price):>10,} ₸".replace(",", " "))
    print(f"похожие:  {int(final[i, 0]):>10,} – {int(final[i, 2]):,} ₸ (медиана {int(final[i, 1]):,})".replace(",", " "))
    print(f"вердикт:  {verdict(row.price, *final[i], n)}")
    print(f"\nаналоги ({how}, найдено {n}):")
    print(comps.to_string(index=False) if len(comps) else "  нет")


show_verdict(int(seg.assign(over=y - final[:, 2]).over.idxmax()))   # самое «переоценённое»
print("\n" + "=" * 90 + "\n")
show_verdict(int(seg[seg.hit].sample(1, random_state=0).index[0]))  # обычное, попавшее в коридор

In [ ]:
OUT = Path("artifacts")
OUT.mkdir(exist_ok=True)
model_full.save_model(OUT / "price_model.cbm")
(OUT / "price_model.json").write_text(json.dumps({
    "features": FEATURE_SETS[BEST_SET],
    "cat_features": cats_full,
    "quantiles": list(QUANTILES),
    "conformal_q": float(np.mean(q_by_fold)),      # средняя поправка по фолдам
    "target": "log(price), ₸ в месяц",
    "metrics_oof": metrics(y, preds["+ конформная калибровка"]),
    "n_train": int(len(df)),
}, ensure_ascii=False, indent=2))
print("сохранено:", *[p.name for p in OUT.iterdir()])

## Что дальше

- **Вердикт и аналоги** уходят в MCP-инструменты `estimate_price` и `get_comparables`,
  оттуда — в агента и на сайт.
- **RAG-объяснение**: те же аналоги отдаются LLM, она пишет, почему цена высокая.
- **Признак «качество ремонта» по фото** (DINOv3 + linear probing на `renovation_text`) —
  единственный крупный признак, которого сейчас не хватает: в колонках нет ни года
  постройки, ни типа дома, а ремонт есть только у 36%.